# NL 03 · B+ 网格

运行 `nl.grid_skeleton.derive`, `nl.grid.derive`（要求 `02_regions_stations.ipynb` 已完成）：按 `bounded_equal_area_budget_v1` 为每个分析区域生成等面积预算网格，写 `data/datasets/2_derived/nl/grid_bplus/bundles/<region>/`。随后展示网格参数、bundle schema、逐区域概况与标准图。

In [ ]:
from sglib.core.infra.paths import find_repo_root
from sglib.dataoverview import stage
from sglib.dataoverview.overview import figures, tables
from sglib.dataoverview.reader import canonical_paths, read_artifact
import pandas as pd

ROOT = find_repo_root()
COUNTRY = "nl"
config = stage.country_config(ROOT, COUNTRY)
context = stage.country_context(ROOT, COUNTRY)
FIGURES = stage.figures_root(ROOT, COUNTRY)
LABEL = config.country_profile.label
GROUP_KEY = config.values["regions"].get("group_key")
GROUPS = [str(item["id"]) for item in config.values["regions"]["items"]]
contract = config.values["station_contract"]
SOURCE_KEY = str(contract["source_key"])
REGION_COLUMN = str(contract["region_column"])

from sglib.dataoverview.processing.features.grid_bundle import load_grid_bundle
report = stage.run_step(ROOT, COUNTRY, ["grid_skeleton", "grid"])
report

## 1 · 来源与许可（source_and_license）

网格是共享处理产物，没有外部数据集；参数全部来自 `general.toml` 的 `[grid]`。

In [ ]:
tables.product_ledger(config.values, products=["grid_skeleton", "grid"])

In [ ]:
pd.Series(config.values["grid"], name="value").to_frame()

## 2 · Schema

正式合同在 `../schemas/grid_bundle.toml`。下面是首个分析区域 bundle 的实际列与元数据。

In [ ]:
REGION = GROUPS[0]
grid, step, metadata = load_grid_bundle(REGION, context.grid_root)
tables.schema_table({f"grid_points[{REGION}]": grid})

In [ ]:
pd.Series({key: value for key, value in metadata.items() if not isinstance(value, (list, dict))}, name="value").to_frame()

## 3 · 覆盖统计（coverage_stats）

In [ ]:
profile = tables.grid_profile(context)
profile

In [ ]:
profile[["n_cells", "n_sources", "area_km2"]].sum().to_frame("total")

## 4 · 单位与时点（unit_and_vintage）

面积在国别等面积 CRS 计算，网格在 generation CRS 生成、以 storage CRS 存储。

In [ ]:
tables.unit_and_vintage(config).query("section == 'crs'")

## 5 · 标准图（standard_figures）

In [ ]:
fig = figures.plot_grid_profile(profile, target_points=int(config.values["grid"]["target_points"]), title=f"{LABEL} B+ grid: step length and realised cell count")
figures.save_figure(fig, FIGURES / "03_grid_profile.png")
fig

In [ ]:
paths = canonical_paths(ROOT, COUNTRY, configuration=config.values)
regions = read_artifact(paths["regions"])
outline = regions if GROUP_KEY is None else regions.loc[regions[GROUP_KEY].astype(str) == REGION]
fig = figures.plot_grid_points(grid, source_key=SOURCE_KEY, region=REGION, outline=outline)
figures.save_figure(fig, FIGURES / f"03_grid_points_{REGION}.png")
fig

下一步：`04_features.py`（长时间批处理，命令行运行）。